# **Diffusion Models: Schedules, Reverse Process, and the Reparameterization Trick**

Notebook 01 showed how a fixed Gaussian chain destroys image information. Now we derive how a model can reverse that chain. The central result is that a neural network can predict the noise in $x_t$; a simple algebraic update then removes a small portion of it.

**Learning goals**
- Compare linear and cosine noise schedules through $\bar\alpha_t$ and SNR.
- Derive the exact posterior $q(x_{t-1}\mid x_t,x_0)$.
- Understand why the reverse transition is learned and how the reparameterization trick makes a Gaussian sample differentiable.
- Derive the simplified noise-prediction loss used by DDPMs.

## 1. Schedule choices

A noise schedule is the sequence $\beta_1,\ldots,\beta_T$. Too much noise in a step destroys detail abruptly; too little wastes steps. The schedule is a design choice that determines the signal-to-noise ratio at every noise level.

The linear schedule often introduced first makes $\beta_t$ increase linearly. A cosine schedule instead designs $\bar\alpha_t$ to decay approximately as a shifted cosine squared, then obtains $\beta_t=1-\bar\alpha_t/\bar\alpha_{t-1}$. Comparing the cumulative curves is more informative than comparing $\beta_t$ alone, because $\bar\alpha_t$ describes total signal retained after all preceding steps.

The schedules below are educational. For a trained model, the schedule used at training and sampling must match.

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

T = 300
def linear_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

def cosine_schedule(T, s=0.008, max_beta=0.999):
    steps = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos(((steps / T + s) / (1 + s)) * math.pi / 2).square()
    abar = f / f[0]
    betas = 1 - abar[1:] / abar[:-1]
    return betas.clamp(max=max_beta).float()

betas_linear = linear_schedule(T)
betas_cosine = cosine_schedule(T)
abar_linear = torch.cumprod(1 - betas_linear, dim=0)
abar_cosine = torch.cumprod(1 - betas_cosine, dim=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name, beta, abar in [('linear', betas_linear, abar_linear), ('cosine', betas_cosine, abar_cosine)]:
    axes[0].plot(beta, label=name); axes[1].plot(abar, label=name)
axes[0].set(title=r'Per-step variance $\beta_t$', xlabel='timestep', ylabel='variance')
axes[1].set(title=r'Cumulative signal $\bar{\alpha}_t$', xlabel='timestep', ylabel='signal coefficient')
for ax in axes: ax.grid(alpha=.25); ax.legend()
plt.tight_layout();

## 2. The reverse transition and the posterior trick

The reverse chain is written $p_\theta(x_{t-1}\mid x_t)$. Ideally, this conditional would map the noisy state toward a clean sample. It is unknown because the data distribution is unknown. However, if we additionally knew the original $x_0$, Bayes' rule and Gaussian algebra give a tractable posterior:

$$q(x_{t-1}\mid x_t,x_0)=\mathcal N(\tilde\mu_t(x_t,x_0),\tilde\beta_t I),$$
$$\tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t},\qquad\tilde\mu_t=\frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0+\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t.$$

The derivation is a product of two Gaussians as functions of $x_{t-1}$: the forward likelihood $q(x_t\mid x_{t-1})$ and the marginal prior $q(x_{t-1}\mid x_0)$. Their precisions (inverse variances) add, so the posterior remains Gaussian. This posterior is an *oracle*: during generation $x_0$ is unavailable. The model's job is to replace the missing information with a learned estimate.

For $t=1$, the posterior variance is zero: the final step returns the predicted clean image without adding fresh noise.

## 3. Reparameterization: move the randomness outside the function

A Gaussian draw with mean $\mu_\theta$ and standard deviation $\sigma_\theta$ can be written

$$x=\mu_\theta+\sigma_\theta\epsilon,\quad\epsilon\sim\mathcal N(0,I).$$

The random variable $\epsilon$ is independent of the learnable parameters. Conditional on a sampled $\epsilon$, the expression is an ordinary differentiable function of $\mu_\theta$ and $\sigma_\theta$. This is the reparameterization trick: randomness remains, while gradients can pass through the deterministic transformation.

In DDPM training the sampled noise is a target, not a latent random variable whose gradient we need. Still, this same Gaussian parameterization explains sampling from the learned reverse transition and why the stochastic part can be represented separately.

In [ ]:
# A tiny gradient demonstration
mu = torch.tensor(1.0, requires_grad=True)
log_sigma = torch.tensor(-0.2, requires_grad=True)
epsilon = torch.randn(())
sample = mu + torch.exp(log_sigma) * epsilon
loss = (sample - 2.0).square()
loss.backward()
print('epsilon:', epsilon.item())
print('sample:', sample.item())
print('d loss / d mu:', mu.grad.item())
print('d loss / d log_sigma:', log_sigma.grad.item())

## 4. The noise-prediction parameterization

From the forward marginal,
$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,$$
we can solve for $x_0$ if the noise is known:
$$x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\epsilon}{\sqrt{\bar\alpha_t}}.$$

A neural network $\epsilon_\theta(x_t,t)$ predicts that noise, conditioned on the noisy image and timestep. Substitute this estimate into the oracle posterior mean. After simplification, the mean becomes

$$\mu_\theta(x_t,t)=\frac{1}{\sqrt{\alpha_t}}\left(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(x_t,t)\right).$$

Thus each denoising step is not mysterious: predict the residual and apply known schedule-dependent coefficients. The network's timestep input matters because the same pixel pattern can mean different things at different noise levels.

## 5. Why the objective becomes mean squared noise error

The variational lower bound for the data likelihood decomposes across timesteps into KL divergences between the true posterior and the model's reverse Gaussian. If the model uses the posterior variance, minimizing each KL reduces to matching means. Because the mean is affine in the predicted noise, this becomes a weighted squared error between true and predicted noise. A commonly used simplified objective drops timestep-dependent weights:

$$\mathcal L_{\mathrm{simple}}=\mathbb E_{x_0,t,\epsilon}\left[\|\epsilon-\epsilon_\theta(\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,t)\|_2^2\right].$$

Each training example therefore requires one clean image, one uniformly sampled timestep, one Gaussian noise tensor, and one forward pass. MSE is averaged over batch and pixels. The random timestep makes the network practice denoising at all corruption levels.

> **Terminology:** The original variational objective contains additional weighting and boundary terms. The simplified objective is a practical parameterization of the denoising problem, not a claim that every likelihood term is identical to unweighted MSE.

In [ ]:
def ddpm_loss(model, x0, alpha_bars, T):
    batch_size = x0.shape[0]
    t = torch.randint(0, T, (batch_size,), device=x0.device)
    noise = torch.randn_like(x0)
    a_bar = alpha_bars.to(x0.device).gather(0, t).view(batch_size, 1, 1, 1)
    xt = a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * noise
    predicted_noise = model(xt, t)
    return torch.nn.functional.mse_loss(predicted_noise, noise)

print('Loss recipe defined; the U-Net is introduced in notebook 03.')

## 6. Reverse sampling in code form

Start with $x_T\sim\mathcal N(0,I)$. For $t=T,\ldots,1$, evaluate $\epsilon_\theta(x_t,t)$, update with the learned mean, and add Gaussian noise with variance $\tilde\beta_t$ except at the last step. This sequential dependency explains why the original DDPM sampler needs many model evaluations.

A sampler must share the model's training schedule, prediction target, image normalization, and timestep convention. A mismatch in any of these changes the scale or interpretation of the network output and usually yields poor images. Notebook 03 implements this loop end to end.

## Summary

- The schedule shapes the signal-to-noise trajectory; compare $\bar\alpha_t$, not only per-step $\beta_t$.
- Knowing $x_0$ gives an exact Gaussian posterior, but sampling does not know $x_0$.
- Predicting noise gives a direct expression for the reverse mean.
- The simplified training target is the injected Gaussian noise, with MSE loss.
- Notebook 03 implements a time-conditioned U-Net, a configurable training loop, and ancestral sampling.